# Conformal geometry

By default, Wakis assigns one material per mesh cell (**staircase** geometry). A curved surface is then replaced by the faces of the cells it passes through, which changes the geometry by up to one cell size. For curved conductors and dielectrics this shifts resonance frequencies and limits how fast results converge with mesh refinement.

With `geometry_mode="conformal"` Wakis instead intersects the STL surfaces with the grid (conformal FIT):

* **PEC walls cut the mesh**: an edge cut by a PEC wall keeps only its length inside the vacuum or dielectric, and a cut face keeps only its area there (a Dey-Mittra-type cut-cell scheme).
* **Dielectric and magnetic interfaces** use effective materials: a mesh edge crossing an interface gets a mix of the two materials, weighted by how the interface cuts it.

This notebook shows both on a pillbox cavity with beam pipes and a dielectric coating on its outer wall:

| | |
|---|---|
| `geometry_mode` | `"staircase"` (default) or `"conformal"` |
| `stl_material_types` | `"normal"`, `"pec"` (and `"sibc"`) per STL solid |
| `bg_material_type` | type of the background material |
| `conformal_stabilization` | `"area_floor"` keeps the time step stable with small cut cells |
| `solver.compute_stable_timestep()` | exact stability limit of the time step |
| `solver.plot_mesh_materials()` | the mesh, its materials and the STL geometry in one plane |

In [1]:
import tempfile
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pyvista as pv
from scipy.constants import c
from scipy.optimize import brentq
from scipy.special import jn_zeros, jv, yv

from wakis import GridFIT3D, SolverFIT3D, WakeSolver

# Show warnings as 'UserWarning: message', without the source file location.
warnings.formatwarning = lambda message, category, *args, **kwargs: (
    f"{category.__name__}: {message}\n"
)

## Geometry

A pillbox cavity (radius $R$, length $L$) with a beam pipe (radius $a$) on each side, built from cylinders as in notebook 001e. The cylindrical wall of the cavity carries a dielectric coating ($\varepsilon_r = 3$, thickness $t$). We describe it with two solids: the coating, a cylinder of radius $R$, and the vacuum, the cavity of radius $b = R - t$ merged with the pipes. Where solids overlap, the solid listed later in `stl_solids` wins, so the vacuum is listed last. The background is PEC: it forms the cavity wall behind the coating, the end walls and the pipe walls. As in notebook 001, the simulation domain is the bounding box of the solids, so the pipes end at the domain boundary.

In [2]:
R, L, t = 0.10, 0.10, 0.0075  # cavity radius, length, coating thickness [m]
a, L_pipe = 0.0125, 0.10  # beam pipe radius and length on each side [m]
b = R - t  # inner radius of the coating
eps_coating = 3.0

pipe = pv.Cylinder(radius=a, height=L + 2 * L_pipe, direction=(0, 0, 1), resolution=100)
cavity = pv.Cylinder(radius=b, height=L, direction=(0, 0, 1), resolution=200)
coating = pv.Cylinder(radius=R, height=L, direction=(0, 0, 1), resolution=200)

stl_folder = tempfile.mkdtemp()
stl_solids = {
    "coating": f"{stl_folder}/coating.stl",
    "vacuum": f"{stl_folder}/vacuum.stl",
}
coating.triangulate().save(stl_solids["coating"])
(pipe.triangulate() | cavity.triangulate()).save(stl_solids["vacuum"])
stl_materials = {"coating": [eps_coating, 1.0, 0.0], "vacuum": "vacuum"}

xmin, xmax, ymin, ymax, zmin, zmax = (pipe + coating).bounds

### Analytic reference

Without the pipes, the azimuthally symmetric TM modes of the coated pillbox are exact Bessel solutions. With $\beta = p\pi/L$, the radial wave numbers are $q_0^2 = k^2 - \beta^2$ in vacuum and $q_1^2 = \varepsilon_r k^2 - \beta^2$ in the coating:

$$E_z = J_0(q_0 r) \quad (r < b), \qquad E_z = J_0(q_1 r)\,Y_0(q_1 R) - Y_0(q_1 r)\,J_0(q_1 R) \quad (b < r < R),$$

which vanishes at the PEC wall. $E_z$ and $H_\varphi \propto (\varepsilon_r / q^2)\, \partial_r E_z$ must be continuous at $r = b$; the roots of this condition are the TM$_{01p}$ frequencies. For TM011, $q_0$ becomes imaginary (the field decays towards the axis); complex Bessel arguments cover that case.

The beam pipes open the end walls, which shifts the real resonance frequencies slightly upwards, so this is a close reference, not an exact one. We look at two modes:

* **TM010** has only $E_z$, parallel to both surfaces. It mainly tests the curved PEC wall.
* **TM011** has a radial electric field that crosses the coating surface, so it also tests the effective materials at the dielectric interface.

In [3]:
def f_tm01p(p):
    # TM01p frequency of the coated pillbox without beam pipes.
    beta = p * np.pi / L

    def mismatch(f):
        k = 2 * np.pi * f / c
        q0 = np.sqrt(complex(k**2 - beta**2))
        q1 = np.sqrt(complex(eps_coating * k**2 - beta**2))
        ez_in, hphi_in = jv(0, q0 * b), -jv(1, q0 * b) / q0
        ez_out = jv(0, q1 * b) * yv(0, q1 * R) - yv(0, q1 * b) * jv(0, q1 * R)
        hphi_out = -(eps_coating / q1) * (
            jv(1, q1 * b) * yv(0, q1 * R) - yv(1, q1 * b) * jv(0, q1 * R)
        )
        return (ez_in * hphi_out - hphi_in * ez_out).real

    # Bracket: between the fully filled and the empty cavity
    f_empty = np.hypot(jn_zeros(0, 1)[0] / R, beta) * c / (2 * np.pi)
    f_filled = max(
        f_empty / np.sqrt(eps_coating),
        1.001 * beta * c / (2 * np.pi * np.sqrt(eps_coating)),
    )
    return brentq(mismatch, f_filled, f_empty)


f_analytic = {"TM010": f_tm01p(0), "TM011": f_tm01p(1)}
for mode, f in f_analytic.items():
    print(f"{mode}: {f / 1e6:.1f} MHz")

TM010: 1145.5 MHz
TM011: 1816.0 MHz


## Grid and solver setup

The conformal grid needs to know how each solid takes part in the cut-cell treatment:

* `stl_material_types`: `"normal"` for vacuum, dielectric or magnetic solids (they are kept and get effective materials at their interfaces), `"pec"` for perfect conductors (they cut the mesh). Here both solids are `"normal"`.
* `bg_material_type="pec"`: the background around the solids is a perfect conductor, so the cavity, end and pipe walls are PEC walls cutting the mesh. It must match the solver background `bg="pec"`.
* `conformal_stabilization="area_floor"`: explained in the stability section below.

The grid nodes are classified with `stl_method="voxelize_rectilinear"`, which needs a mesh with uniform spacing along each axis. We start with a coarse mesh of 1 cm cells, larger than the coating thickness:

In [4]:
mesh = dict(
    xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax, zmin=zmin, zmax=zmax,
    Nx=21, Ny=21, Nz=31,  # 1 cm cells
    stl_method="voxelize_rectilinear", verbose=0,
)  # fmt: skip

# Conformal geometry options (see above)
conformal = dict(
    geometry_mode="conformal",
    stl_material_types={"coating": "normal", "vacuum": "normal"},
    bg_material_type="pec",
    conformal_stabilization="area_floor",
)

# Closed structure: PEC background and PEC domain boundaries
solver_setup = dict(
    bg="pec", bc_low=["pec"] * 3, bc_high=["pec"] * 3,
    use_stl=True, dtype=np.float64, verbose=0,
)  # fmt: skip

grid_coarse = GridFIT3D(
    **mesh, stl_solids=stl_solids, stl_materials=stl_materials, **conformal
)
solver_coarse = SolverFIT3D(grid_coarse, **solver_setup)

Generating grid with 13671 mesh cells...


234 edges contain material layers thinner than the mesh resolution; at most one interface per edge is represented. Refine the mesh there if these layers matter.


Initializing Electromagnetic solver...


## Inspecting the geometry on the mesh

`solver.plot_mesh_materials()` shows one grid plane with the mesh, the material of every face and the exact STL cross-section (black lines). In FIT, the material of an electric or magnetic DOF belongs to the face its edge pierces, so there are two views:

* `grid_type="primal"`: primal faces, coloured by $\mu_r$. Faces cut by a PEC wall are coloured only in their part inside the cavity (the PEC part is grey), and the mesh edges stop at the wall.
* `grid_type="dual"`: dual faces, coloured by $\varepsilon_r$ (or $\sigma$ with `quantity="sigma"`). Dual faces are never cut; faces whose electric edge lies in PEC are grey.

The plane normal selects the tensor component: in an $x$-normal plane the faces carry $\varepsilon_{r,x}$. Here the coarse mesh, in the middle of the cavity (left) and through the axis (right):

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
solver_coarse.plot_mesh_materials(grid_type="primal", normal="z", ax=axes[0])
solver_coarse.plot_mesh_materials(grid_type="dual", normal="x", ax=axes[1])
fig.tight_layout()

## Geometry diagnostics

While building a conformal grid, Wakis checks every mesh edge against the STL surfaces and warns about two kinds of edges:

* **Thin feature**: the edge contains material structure thinner than the mesh, e.g. a layer that lies completely between the two edge ends. At most one interface per edge is represented, so the layer is partly or completely lost.
* **Inconsistent**: the STL crossings found on the edge do not match the inside/outside classification of its end points, e.g. at an open (non-watertight) STL surface. These edges get no or only an ambiguous conformal correction.

The warning only gives the number of edges. `plot_mesh_materials()` shows where they are: orange circles mark thin features, magenta crosses inconsistent edges, and the legend counts them in the plane.

On the coarse mesh, the coating (7.5 mm) is thinner than a cell (10 mm): many edges through the coating contain both of its surfaces, and only one of them can be represented. The orange circles mark these edges, and in the side view the coating reaches only $\varepsilon_r \approx 1.5$ instead of 3. Halving the cell size to 5 mm, below the coating thickness, resolves it: no edge then contains both surfaces. We set up the staircase and the conformal grid on this mesh and use it from here on.

In [6]:
mesh.update(Nx=41, Ny=41, Nz=61)  # 5 mm cells
grids = {
    "staircase": GridFIT3D(**mesh, stl_solids=stl_solids, stl_materials=stl_materials),
    "conformal": GridFIT3D(
        **mesh, stl_solids=stl_solids, stl_materials=stl_materials, **conformal
    ),
}
solvers = {mode: SolverFIT3D(grid, **solver_setup) for mode, grid in grids.items()}

Generating grid with 102541 mesh cells...
Generating grid with 102541 mesh cells...


Initializing Electromagnetic solver...
Initializing Electromagnetic solver...


No warning now. **PEC wall.** The primal $z$-plane in the middle of the cavity: the staircase follows the cell faces, the conformal mesh follows the cavity wall (outer circle).

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (mode, solver) in zip(axes, solvers.items()):
    solver.plot_mesh_materials(grid_type="primal", normal="z", ax=ax)
    ax.set_title(f"{mode}: " + ax.get_title())
fig.tight_layout()

**Dielectric interface.** The dual $z$-plane in the middle of the cavity shows $\varepsilon_{r,z}$ in the cavity and the coating ring. The staircase switches between 1 and 3 from one cell to the next; the conformal mesh gives the edges crossing the inner surface of the coating an effective value in between. It depends on where the surface cuts the edge and on its orientation: the field normal to the surface sees the harmonic mean of the two materials (series), the field along it the arithmetic mean (parallel).

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (mode, solver) in zip(axes, solvers.items()):
    solver.plot_mesh_materials(grid_type="dual", normal="z", ax=ax)
    ax.set_title(f"{mode}: " + ax.get_title())
fig.tight_layout()

Red outlines mark the conformal stabilization (see below). In Jupyter, `interactive=True` adds buttons for the plane normal (and the material) and a slider to move through the planes; it needs an interactive backend such as `%matplotlib widget`:

```python
solvers["conformal"].plot_mesh_materials(grid_type="dual", interactive=True)
```

**Inconsistent edges: an STL with a hole.** If the coating STL had a hole in its outer wall, e.g. from a damaged CAD export, Wakis would warn that the surface is not closed and mark the edges where the grid lines pass through the gap:

In [9]:
coating_hole = coating.triangulate().subdivide(2, subfilter="linear")
centers = coating_hole.cell_centers().points
angle = np.degrees(np.arctan2(centers[:, 1], centers[:, 0]))
hole = (
    (np.abs(np.hypot(centers[:, 0], centers[:, 1]) - R) < 1e-3)  # outer wall
    & (np.abs(angle - 30) < 12)
    & (np.abs(centers[:, 2]) < 0.02)
)
stl_solids_hole = {**stl_solids, "coating": f"{stl_folder}/coating_hole.stl"}
coating_hole.extract_cells(np.nonzero(~hole)[0]).extract_surface(
    algorithm="dataset_surface"
).save(stl_solids_hole["coating"])

grid_hole = GridFIT3D(
    **mesh, stl_solids=stl_solids_hole, stl_materials=stl_materials, **conformal
)
fig, ax = plt.subplots(figsize=(8, 6))
SolverFIT3D(grid_hole, **solver_setup).plot_mesh_materials(
    grid_type="primal", normal="z", ax=ax
)

Generating grid with 102541 mesh cells...


112 edges have STL crossings that disagree with the grid point classification and receive no or an ambiguous conformal correction. Check that the STL solids are closed and not tangent to grid points.


Initializing Electromagnetic solver...


(<Figure size 800x600 with 2 Axes>,
 <Axes: title={'center': 'primal mesh, z = 0 m (index 30)'}, xlabel='x [m]', ylabel='y [m]'>)

These edges get no conformal cut and take their material from their end points, so the geometry falls back to a staircase at the hole, while it stays conformal elsewhere. Repair such an STL (make it watertight) before simulating.

## Stability of the time step

The explicit leapfrog update is stable only for $\Delta t < 2/\omega_{\max}$, where $\omega_{\max}$ is the highest eigenfrequency of the discrete system. On an uncut grid this is the CFL limit. A face cut by a PEC wall into a tiny piece makes $\omega_{\max}$ very large, and the time step needed for stability would become tiny.

`conformal_stabilization="area_floor"` raises every cut face to at least a fraction (default 0.1) of its full area, and removes edges with less than that fraction left in the cavity. This bounds the time step reduction at the cost of a small, local geometry change. `compute_stable_timestep()` computes the exact stability limit of the assembled solver and compares it with the time step in use:

In [10]:
cases = {
    "staircase": solvers["staircase"],
    "conformal, no stabilization": SolverFIT3D(
        GridFIT3D(
            **mesh,
            stl_solids=stl_solids,
            stl_materials=stl_materials,
            **{**conformal, "conformal_stabilization": None},
        ),
        **solver_setup,
    ),
    "conformal, area floor": solvers["conformal"],
}
for name, solver in cases.items():
    result = solver.compute_stable_timestep(verbose=False)
    print(
        f"{name:28s} dt_max / dt_CFL = {result['dt_max'] / result['dt_cfl']:.3g}   "
        f"stable: {result['stable']}"
    )
print(
    f"area floor: {solvers['conformal'].n_area_floored} faces raised, "
    f"{solvers['conformal'].n_edges_eliminated} edges removed"
)

Generating grid with 102541 mesh cells...


Initializing Electromagnetic solver...


staircase                    dt_max / dt_CFL = 1   stable: True


conformal, no stabilization  dt_max / dt_CFL = 0.857   stable: True


conformal, area floor        dt_max / dt_CFL = 0.876   stable: True
area floor: 1036 faces raised, 304 edges removed


Without stabilization, the cut faces lower the stable time step here to about 86 % of the CFL limit. The solver's default time step (half the CFL limit, `cfln=0.5`) is still below it, but smaller cut faces, for example a surface passing very close to a grid node, lower it further until the default time step diverges. The area floor bounds this reduction for any geometry; here it keeps the stable time step at about 88 % of the CFL limit.

## Wakefield simulation

We run the same wakefield simulation on both grids: a Gaussian bunch on axis, 20 m of wake. PEC walls make the modes undamped, so the impedance shows a peak at each cavity mode; the side lobes come from cutting the wake after 20 m. The simulations print progress bars; `%%capture` keeps the notebook short.

In [11]:
%%capture
impedance = {}
for mode, grid in grids.items():
    folder = tempfile.mkdtemp()
    wake = WakeSolver(
        q=1e-9,
        sigmaz=0.03,
        beta=1.0,
        save=False,
        results_folder=f"{folder}/",
        Ez_file=f"{folder}/Ez.h5",
        verbose=0,
    )
    solver = SolverFIT3D(grid, wake, **solver_setup)
    solver.wakesolve(wakelength=20.0, plot=False)
    wake.calc_long_Z(samples=20001, fmax=2.5e9)
    impedance[mode] = (np.asarray(wake.f), np.abs(wake.Z))

In [12]:
fig, ax = plt.subplots(figsize=(10, 4.5))
for mode, (f, Z) in impedance.items():
    ax.plot(f / 1e9, Z, label=mode)
for name, f_exact in f_analytic.items():
    ax.axvline(f_exact / 1e9, color="k", ls="--", lw=0.8)
    ax.text(f_exact / 1e9, 2e4, f" {name}", fontsize=9)
ax.set(
    xlabel="f [GHz]",
    ylabel=r"$|Z_\parallel|$ [$\Omega$]",
    yscale="log",
    xlim=(0.8, 2.2),
    ylim=(10, 5e4),
)
ax.legend()

# Impedance peak next to each analytic frequency
for mode, (f, Z) in impedance.items():
    peaks = []
    for name, f_exact in f_analytic.items():
        window = np.abs(f - f_exact) < 0.15 * f_exact
        f_peak = f[window][np.argmax(Z[window])]
        peaks.append(
            f"{name} {f_peak / 1e6:.1f} MHz ({100 * (f_peak / f_exact - 1):+.2f} %)"
        )
    print(f"{mode:10s}: " + ", ".join(peaks))

staircase : TM010 1129.2 MHz (-1.43 %), TM011 1774.4 MHz (-2.29 %)
conformal : TM010 1147.9 MHz (+0.21 %), TM011 1820.4 MHz (+0.25 %)


On the same mesh, the staircase geometry shifts TM010 and TM011 by $-1.4$ % and $-2.3$ %, while the conformal frequencies lie about $0.2$ % above the analytic values. Most of this remaining difference comes from the beam pipes, which the analytic reference leaves out. The accuracy of the conformal scheme itself is checked against exact cavity modes in the Wakis tests (`tests/test_020_conformal.py`), including its convergence with mesh refinement.

## Summary

* `geometry_mode="conformal"` represents curved PEC walls and dielectric interfaces inside the mesh cells. Here it brings the mode frequencies close to the analytic values on a mesh where the staircase geometry is off by one to two percent.
* Each STL solid needs a `stl_material_types` entry: `"normal"` solids get effective materials at their interfaces, `"pec"` solids cut the mesh. `bg_material_type` sets the type of the background and must match the solver background `bg`.
* Check the geometry diagnostics warnings: `plot_mesh_materials()` marks where layers thinner than the mesh are lost (refine the mesh until they span a cell) and where the STL is not consistent (e.g. not watertight).
* Keep `conformal_stabilization="area_floor"` for time-domain simulations with PEC cut cells; `compute_stable_timestep()` checks the resulting time step.
* `plot_mesh_materials()` is the quickest way to check how the mesh represents a geometry before running a simulation.

Current limitations: each mesh edge is cut at most once (an edge crossing two surfaces is reported in a warning and treated approximately), and the effective materials at dielectric interfaces are diagonal approximations of the full tensor.